# cuTile Python: Transpose

In the previous notebook you multiplied two matrices using 2D tiles. Each output tile accumulated partial products along the shared dimension.

This notebook introduces a different kind of kernel: one that moves values without computing new ones. Transpose swaps rows and columns. The values stay the same, only their positions change.

In [ ]:
import os

_install_marker = os.path.expanduser(
    "~/.accelerated-computing-hub-installed"
)

# Bootstrap dependencies only in Google Colab.
if os.getenv("COLAB_RELEASE_TAG") and not os.path.exists(_install_marker):
  try:
    import cuda.tile
    import cupy
    from numba import cuda
  except ImportError:
    print("Installing PIP packages.")
    !pip install --upgrade "cuda-tile[tileiras]==1.4.0" "cuda-toolkit==13.2.1" "cuda-python==13.2.0" "cupy-cuda13x==14.2.0" "numba==0.63.1" "numba-cuda[cu13]==0.30.4" > /dev/null 2>&1
  open(_install_marker, "a").close()

In [ ]:
import cuda.tile as ct
import cupy as cp

## Rearranging Without Computing

Every kernel so far loaded tiles, computed new values, and stored the results. Transpose is different: the output values are exactly the same as the input. What changes is where they go.

Element at row `i`, column `j` moves to row `j`, column `i` in the output. A 4x8 matrix becomes an 8x4 matrix.

Two details determine where each transposed tile goes:

The output array has its dimensions swapped. An `M x N` input produces an `N x M` output.

The store index becomes `(col, row)` instead of `(row, col)`, which places the tile in the correct output position. But the elements inside the tile also need to be rearranged. A tile of shape `(tm, tn)` needs to become `(tn, tm)` before it fits there. `ct.transpose(tile)` does that.

For example, say `IN` is a 4×6 matrix with `tm=2`, `tn=3`:

```
 1   2   3   4   5   6
 7   8   9  10  11  12
13  14  15  16  17  18
19  20  21  22  23  24
```

Block `(row=0, col=1)` loads rows 0–1, cols 3–5:

```
 4   5   6
10  11  12
```

After `ct.transpose`, the tile becomes a 3×2 block:

```
 4  10
 5  11
 6  12
```

It is then stored at index `(col=1, row=0)` in the output, placing it at rows 3–5, cols 0–1 of `OUT`.

## Example: Transpose

Each block handles one `(tm, tn)` tile: it loads the tile at `(row, col)`, transposes it to shape `(tn, tm)` using `ct.transpose`, and writes it to position `(col, row)` in the output. With M=1024 and tm=32, you need 1024/32 = 32 blocks to cover the rows. With N=2048 and tn=64, you need 2048/64 = 32 blocks to cover the columns. The grid is `(32, 32, 1)`, 1024 blocks total.

In [ ]:
@ct.kernel
def transpose(IN: ct.Array, OUT: ct.Array, tm: ct.Constant[int], tn: ct.Constant[int]):
  row = ct.bid(0)
  col = ct.bid(1)

  tile = ct.load(IN, index=(row, col), shape=(tm, tn))

  ct.store(OUT, index=(col, row), tile=ct.transpose(tile))

In [ ]:
M, N = 1024, 2048
tm, tn = 32, 64

IN = cp.random.uniform(-5, 5, (M, N), dtype=cp.float32)
OUT = cp.zeros((N, M), dtype=cp.float32)

grid = (ct.cdiv(M, tm), ct.cdiv(N, tn), 1)
print(f"Grid: {grid[0]} × {grid[1]} = {grid[0] * grid[1]:,} blocks")

ct.launch(cp.cuda.get_current_stream(), grid, transpose, (IN, OUT, tm, tn))

cp.testing.assert_array_almost_equal(OUT, IN.T)
print("Transpose OK")

## Exercise: Symmetric Sum

A symmetric matrix satisfies `A[i][j] == A[j][i]`. One way to produce a symmetric matrix from any square input is to add it to its own transpose: `OUT = IN + IN.T`.

Write a kernel that computes this. Each block should load the tile at `(row, col)` and the tile at `(col, row)`, add them with the second tile transposed, and store the result at `(row, col)` in the output.

Use a square matrix and square tiles so both loads have the same shape.

Complete the TODO below, then uncomment the validation code to check your work.

In [ ]:
@ct.kernel
def symmetric_sum(IN: ct.Array, OUT: ct.Array, t: ct.Constant[int]):
  # TODO: load the tile at (row, col) and the tile at (col, row).
  # Add them together, transposing the second one, and store at (row, col).
  pass


# N = 1024
# t = 32
# IN = cp.random.uniform(-5, 5, (N, N), dtype=cp.float32)
# OUT = cp.zeros_like(IN)
# grid = (ct.cdiv(N, t), ct.cdiv(N, t), 1)
# ct.launch(cp.cuda.get_current_stream(), grid, symmetric_sum, (IN, OUT, t))
# cp.testing.assert_array_almost_equal(OUT, IN + IN.T)
# print("Exercise OK")

A tiled transpose swaps the output dimensions, swaps the block coordinates to `(col, row)` when storing, and passes each tile through `ct.transpose`. Every element lands in its correct output position.